# Ingestão Bronze — Sismos Globais (USGS)

Primeira etapa do projeto **global-seismic-analytics-lakehouse** (arquitetura medalhão bronze/silver/gold).

Este notebook busca terremotos globais na API pública do USGS (FDSN event query), achata o GeoJSON em um DataFrame Spark e grava na tabela bronze `workspace.sismos_bronze.eventos_usgs` em modo *append*, com colunas de auditoria (`_ingested_at`, `_source`).

In [0]:
# ---------------------------------------------------------------
# PARÂMETROS DA INGESTÃO
# ---------------------------------------------------------------
from datetime import datetime, timedelta, timezone

# Janela de ingestão (default: últimos 30 dias). Formato: YYYY-MM-DD
_data_fim = datetime.now(timezone.utc).date()
_data_inicio = _data_fim - timedelta(days=30)

STARTTIME = _data_inicio.isoformat()
ENDTIME = _data_fim.isoformat()

# API pública do USGS (FDSN event query) e filtros
API_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
MIN_MAGNITUDE = 2.5
PAGE_LIMIT = 1000      # eventos por página (paginação via offset/limit)
MAX_EVENTS = 20000    # limite de segurança da API por consulta

# Destino da camada bronze no Unity Catalog
CATALOG = "workspace"
SCHEMA = "sismos_bronze"
TABLE = "eventos_usgs"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

print(f"Janela de ingestão : {STARTTIME} até {ENDTIME}")
print(f"Magnitude mínima   : {MIN_MAGNITUDE}")
print(f"Tabela bronze      : {BRONZE_TABLE}")

In [0]:
# ---------------------------------------------------------------
# BUSCA NA API DO USGS (paginação offset/limit + tratamento de erros)
# ---------------------------------------------------------------
import requests


def fetch_usgs_events(starttime: str, endtime: str, minmagnitude: float,
                      page_limit: int, max_events: int):
    """Consulta a API FDSN do USGS paginando com offset/limit.

    Retorna (lista de features GeoJSON, total reportado pela API).
    """
    # ATENÇÃO: o offset da API do USGS é 1-based (valor mínimo = 1)
    events, offset, total_reportado = [], 1, None
    try:
        while True:
            params = {
                "format": "geojson",
                "starttime": starttime,
                "endtime": endtime,
                "minmagnitude": minmagnitude,
                "orderby": "time",
                "limit": page_limit,
                "offset": offset,
            }
            resp = requests.get(API_URL, params=params, timeout=60)
            resp.raise_for_status()  # erro HTTP (4xx/5xx) -> exceção clara
            payload = resp.json()
            features = payload.get("features") or []
            if total_reportado is None:
                total_reportado = int(payload.get("metadata", {}).get("count", len(features)))
            events.extend(features)
            print(f"  Página {(offset - 1) // page_limit + 1}: {len(features)} eventos "
                  f"(acumulado {len(events)} de {total_reportado})")
            if not features or len(events) >= total_reportado or len(events) >= max_events:
                if total_reportado > max_events:
                    print(f"  AVISO: janela com {total_reportado} eventos; "
                          f"limite de segurança de {max_events} atingido.")
                break
            offset += page_limit
    except requests.HTTPError as e:
        raise RuntimeError(f"Erro HTTP na API do USGS (status {resp.status_code}): {resp.text[:300]}") from e
    except requests.RequestException as e:
        raise RuntimeError(f"Falha de rede ao consultar a API do USGS: {e}") from e
    return events, total_reportado


raw_events, total_api = fetch_usgs_events(STARTTIME, ENDTIME, MIN_MAGNITUDE, PAGE_LIMIT, MAX_EVENTS)
print(f"\nEventos retornados pela API: {len(raw_events)} (total reportado: {total_api})")
if not raw_events:
    print("AVISO: nenhum evento retornado para a janela informada. A gravação será pulada.")

In [0]:
# ---------------------------------------------------------------
# PARSE DO GEOJSON -> DATAFRAME SPARK (campos achatados)
# ---------------------------------------------------------------
from pyspark.sql import Row
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType, BooleanType, TimestampType)


def feature_to_row(f: dict):
    """Achata uma feature GeoJSON do USGS em uma Row com os campos de interesse."""
    p = f.get("properties") or {}
    coords = (f.get("geometry") or {}).get("coordinates") or []
    t = p.get("time")
    return Row(
        event_id=f.get("id"),
        magnitude=p.get("mag"),
        mag_type=p.get("magType"),
        place=p.get("place"),
        alert=p.get("alert"),
        significance=p.get("sig"),
        tsunami=bool(p.get("tsunami") or 0),
        event_type=p.get("type"),
        event_time_utc=datetime.fromtimestamp(t / 1000.0, tz=timezone.utc) if t else None,
        latitude=coords[1] if len(coords) > 1 and coords[1] is not None else None,
        longitude=coords[0] if len(coords) > 0 and coords[0] is not None else None,
        depth_km=coords[2] if len(coords) > 2 else None,
        url=p.get("url"),
    )


BRONZE_SCHEMA = StructType([
    StructField("event_id", StringType(), True),
    StructField("magnitude", DoubleType(), True),
    StructField("mag_type", StringType(), True),
    StructField("place", StringType(), True),
    StructField("alert", StringType(), True),
    StructField("significance", IntegerType(), True),
    StructField("tsunami", BooleanType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_time_utc", TimestampType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("depth_km", DoubleType(), True),
    StructField("url", StringType(), True),
])

rows = [feature_to_row(f) for f in raw_events]
bronze_df = spark.createDataFrame(rows, schema=BRONZE_SCHEMA)

print(f"DataFrame bronze criado com {bronze_df.count()} linhas")
bronze_df.printSchema()
bronze_df.show(5, truncate=False)

In [0]:
# ---------------------------------------------------------------
# GRAVAÇÃO NA CAMADA BRONZE (Unity Catalog, Delta, append + auditoria)
# ---------------------------------------------------------------
from pyspark.sql import functions as F

n_eventos = bronze_df.count()

if n_eventos == 0:
    print("Nenhum evento a ingerir — gravação PULADA (regra de qualidade: não ingerir zero eventos).")
    INGESTED_THIS_RUN = 0
else:
    # Cria (ou verifica) o schema dedicado do projeto
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

    (
        bronze_df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source", F.lit(
            f"{API_URL}?format=geojson&starttime={STARTTIME}&endtime={ENDTIME}&minmagnitude={MIN_MAGNITUDE}"))
        .write
        .mode("append")
        .format("delta")
        .saveAsTable(BRONZE_TABLE)
    )
    INGESTED_THIS_RUN = n_eventos
    print(f"{INGESTED_THIS_RUN} eventos gravados em {BRONZE_TABLE} (modo append, com colunas de auditoria).")

In [0]:
# ---------------------------------------------------------------
# VALIDAÇÃO PÓS-INGESTÃO
# ---------------------------------------------------------------
print(f"Eventos ingeridos nesta execução : {INGESTED_THIS_RUN}")

if INGESTED_THIS_RUN == 0:
    print("AVISO: nenhum evento ingerido nesta execução — revise a janela de datas e a disponibilidade da API.")
else:
    tabela_bronze = spark.table(BRONZE_TABLE)
    total_tabela = tabela_bronze.count()
    print(f"Total de linhas na tabela bronze : {total_tabela}")

    # Check de qualidade 1 — magnitudes nulas (nesta execução)
    n_null_mag = bronze_df.filter(F.col("magnitude").isNull()).count()
    print(f"Eventos com magnitude nula (nesta execução): {n_null_mag}"
          + ("   ⚠ ATENÇÃO: revisar antes de promover à silver" if n_null_mag else "   OK"))

    # Check de qualidade 2 — duplicidades de event_id (nesta execução)
    n_distintos = bronze_df.select("event_id").distinct().count()
    print(f"Event_ids distintos (nesta execução)    : {n_distintos}"
          + ("   ⚠ ATENÇÃO: há duplicidade" if n_distintos != INGESTED_THIS_RUN else "   OK"))

    # Amostra das linhas ingeridas agora (mais recentes pela auditoria)
    display(tabela_bronze.orderBy(F.col("_ingested_at").desc(), F.col("event_time_utc").desc()).limit(10))

In [0]:
%sql
-- ============================================================================
-- NOTEBOOK: 00_setup_and_delta_fundamentals.sql
-- OBJETIVO: Configurar a hierarquia do Unity Catalog (catálogo, schemas e volumes)
--           e validar propriedades ACID, Time Travel e Lineage no Delta Lake.
-- ============================================================================

-- ----------------------------------------------------------------------------
-- PARTE 1: DEMONSTRAÇÃO DELTA LAKE (VERSIONAMENTO E TIME TRAVEL)
-- Simula o ciclo de vida de um dado sísmico real (estimativa automática -> revisões)
-- ----------------------------------------------------------------------------

-- 1. Cria a tabela gerenciada no catálogo padrão do workspace (Versão 0)
CREATE TABLE IF NOT EXISTS workspace.default.demo_grandes_2025 (
  id           STRING,
  lugar        STRING,
  magnitude    DOUBLE,
  horario_utc  TIMESTAMP,
  status       STRING
);

-- 2. Insere registros iniciais simulando leituras automáticas e revisadas (Versão 1)
INSERT INTO workspace.default.demo_grandes_2025 VALUES
  ('us6000qw60', 'Kamchatka Peninsula, Russia', 8.0, TIMESTAMP '2025-07-29 23:24:52', 'automatic'),
  ('us7000pn9s', 'Mandalay, Burma (Myanmar)',   7.7, TIMESTAMP '2025-03-28 06:20:52', 'reviewed'),
  ('us6000rgf4', 'Drake Passage',               7.6, TIMESTAMP '2025-10-10 20:29:20', 'reviewed');

-- 3. Simula revisões sismológicas consecutivas no evento de Kamchatka (Versões 2 e 3)
UPDATE workspace.default.demo_grandes_2025 
SET magnitude = 8.7 
WHERE id = 'us6000qw60';

UPDATE workspace.default.demo_grandes_2025 
SET magnitude = 8.8, status = 'reviewed' 
WHERE id = 'us6000qw60';

-- 4. Consulta o log de transações (_delta_log) com usuário, horário UTC e operação
DESCRIBE HISTORY workspace.default.demo_grandes_2025;

-- 5. Time Travel: lê o estado original da tabela antes das revisões (Versão 1)
SELECT id, magnitude, status 
FROM workspace.default.demo_grandes_2025 VERSION AS OF 1;

-- 6. Simula exclusão acidental (DELETE sem WHERE) e aplica rollback via RESTORE
DELETE FROM workspace.default.demo_grandes_2025;

-- Restaura a tabela para o estado íntegro da Versão 3 (gera uma nova versão de auditoria)
RESTORE TABLE workspace.default.demo_grandes_2025 TO VERSION AS OF 3;

-- Valida se os registros foram recuperados com sucesso
SELECT * FROM workspace.default.demo_grandes_2025;


-- ----------------------------------------------------------------------------
-- PARTE 2: ESTRUTURAÇÃO DO UNITY CATALOG (ARQUITETURA MEDALHÃO)
-- Namespace de 3 níveis: catalogo.schema.objeto
-- ----------------------------------------------------------------------------

-- 7. Cria o catálogo dedicado do projeto (executar esta instrução primeiro)
CREATE CATALOG IF NOT EXISTS sismos;

-- 8. Cria os schemas das camadas Bronze, Silver e Gold
CREATE SCHEMA IF NOT EXISTS sismos.bronze;
CREATE SCHEMA IF NOT EXISTS sismos.silver;
CREATE SCHEMA IF NOT EXISTS sismos.gold;

-- 9. Cria os volumes gerenciados para arquivos brutos (landing) e estado do Auto Loader (checkpoints)
CREATE VOLUME IF NOT EXISTS sismos.bronze.landing;
CREATE VOLUME IF NOT EXISTS sismos.bronze.checkpoints;


-- ----------------------------------------------------------------------------
-- PARTE 3: LINHAGEM AUTOMÁTICA (LINEAGE) E GOVERNANÇA
-- ----------------------------------------------------------------------------

-- 10. Gera uma tabela agregada na camada Gold para registrar linhagem automática entre catálogos
CREATE OR REPLACE TABLE sismos.gold.demo_resumo AS
SELECT 
  status, 
  count(*) AS eventos, 
  max(magnitude) AS maior_magnitude
FROM workspace.default.demo_grandes_2025
GROUP BY status;

-- 11. Verifica o usuário autenticado e as permissões atuais do schema Gold
SELECT current_user();
SHOW GRANTS ON SCHEMA sismos.gold;

In [0]:
import requests

r = requests.get(
    "https://earthquake.usgs.gov/fdsnws/event/1/count",
    params={
        "format": "geojson",
        "starttime": "2025-07-01",
        "endtime": "2025-08-01",
        "minmagnitude": 2.5,
    },
    timeout=30,
)
print(r.status_code, r.json())